# CA3

### Imports

Import all necessary packages.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


### Reading data

Construct a relative file path, read a CSV file into a DataFrame, and print the first 5 rows.

In [3]:
current_folder = Path.cwd()
all_train_data_path = current_folder / "data" / "train.csv"
final_test_data_path = current_folder / "data" / "test.csv"

df = pd.read_csv(all_train_data_path, index_col="Id")

df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
Id,,,,,,,,,,,,
13,7.8,0.610,0.29,1.6,0.114,9.0,29.0,0.99740,3.26,1.56,9.1,0
705,8.4,1.035,0.15,6.0,0.073,11.0,54.0,0.99900,3.37,0.49,9.9,0
1408,8.1,0.290,0.36,2.2,0.048,35.0,53.0,0.99500,3.27,1.01,12.4,1
144,5.2,0.340,0.00,1.8,0.050,27.0,63.0,0.99160,3.68,0.79,14.0,1
557,15.6,0.645,0.49,4.2,0.095,10.0,23.0,1.00315,2.92,0.74,11.1,0


### Data exploration and visualisation

Display each column's data type and counts of unique and missing values.

**Observations**
* The dataset contains no missing values.
* All features are numerical.
* The target variable is binary.

In [4]:
target = df["quality"]
df = df.drop(columns=["quality"])

In [ ]:
corr = df.corr()

plt.figure(figsize=(12, 10))

sns.heatmap(
    data=corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
)

plt.title("Feature correlations")
plt.tight_layout()
plt.show()



Visualize the distributions of features using histograms and box plots.

The box plots suggest that the features contain numerous outliers. Let's identify these outliers using the interqurtile range (IQR) method.

### Data cleaning

We have too many potential outliers to remove them all without significantly reducing the size of our dataset. Let's skip data cleaning for now.

### Data preprocessing and visualisation

Let's add new features.

In [1532]:
def add_features(df):
    new_df = df.copy()
    
    new_df["free_2"] = df["free sulfur dioxide"] ** 2
    
    #new_df["fraction sulfur dioxide"] = (df["free sulfur dioxide"] / df["total sulfur dioxide"])

    new_df["bound sulfur dioxide"] = (df["total sulfur dioxide"] - df["free sulfur dioxide"])
    
    return new_df.drop(columns=["pH","density", "residual sugar"])

In [1533]:
df = add_features(df)

In [1534]:
log_features = [
    "chlorides", 
    "free sulfur dioxide", 
    "total sulfur dioxide",
    "bound sulfur dioxide",
    "sulphates",
]

df[log_features] = df[log_features].apply(np.log)

### Modelling

In [1535]:
X = df
y = target

n_iter = 200

# Set ranges for C and gamma
range_C = [0.001]
range_gamma = [0.2]

acc_train_range = np.zeros((len(range_C), len(range_gamma))) 
acc_test_range = np.zeros((len(range_C), len(range_gamma)))

for C_ind, C_val in enumerate(range_C):
    for gamma_ind, gamma in enumerate(range_gamma):

        accuracy_train = np.zeros(n_iter)
        accuracy_test = np.zeros(n_iter)

        for rs in range(n_iter):

            # Split into train and test
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.3, random_state=rs, stratify=y)

            # Scale
            sc = StandardScaler().fit(X_train)
            X_train = sc.transform(X_train)
            X_test = sc.transform(X_test)

            # Train
            clf = svm.SVC(
                kernel="sigmoid",
                #degree=2,
                C=C_val,
                gamma=0.01,
                random_state=42,
                coef0=0
            ).fit(X_train, y_train)

            # Calculate accuracy
            accuracy_train[rs] = clf.score(X_train, y_train)
            accuracy_test[rs] = clf.score(X_test, y_test)

        acc_train_range[C_ind, gamma_ind] = accuracy_train.mean()
        acc_test_range[C_ind, gamma_ind] = accuracy_test.mean()

In [1536]:
best_idx = np.unravel_index(
    np.argmax(acc_test_range),
    acc_test_range.shape,
)

best_C = range_C[best_idx[0]]
best_gamma = range_gamma[best_idx[1]]

print(f"Best average accuracy: {acc_test_range.max():.5f}")
print(f"Parameter C: {best_C:.5f}")
print(f"Parameter gamma: {best_gamma:.5f}")


Best average accuracy: 0.71934
Parameter C: 0.00100
Parameter gamma: 0.20000


In [1537]:
"""
Best average accuracy: 0.75753
Parameter C: 0.80000
Parameter gamma: 0.02500
"""
None

### Final evaluation

In [1538]:
X_train = X
y_train = y

X_test = pd.read_csv(final_test_data_path, index_col="Id")

X_test = add_features(X_test)

X_test[log_features] = X_test[log_features].apply(np.log)

sc = StandardScaler().fit(X_train)
X_train = sc.transform(X_train)
X_test = sc.transform(X_test)

clf = svm.SVC(
    kernel="rbf",
    C=best_C,
    gamma=best_gamma,
    random_state=42
).fit(X_train, y_train)

### Kaggle submission

In [1539]:
y_test = clf.predict(X_test)
y_test = pd.DataFrame(y_test, columns=["quality"])
y_test.index.name = "Id"

predictions_path = current_folder / "data" / "SVM.csv"

y_test.to_csv(predictions_path, index=True)